# Feasibility of EODAG

In [ ]:
import helper.maps as helper 
import helper.harvester_code as harvester

from eodag import EODataAccessGateway
dag = EODataAccessGateway()

### Copernicus Dataspace Ecosystem
#### 1. Searching Data

In [ ]:
cop_products = dag.search(
    provider="cop_dataspace",
    collection="S2_MSI_L2A",
    start="2026-04-20T11:00:00",
    end="2026-04-21T12:00:00",
    #geom="POLYGON((3 55, 3 47, 18 47, 18 55, 3 55))",
    **{"grid:code": "MGRS-32UPU",},
    limit=1000
)

cop_products

In [ ]:
folium_map_cop = helper.map_results(False, cop_products)
folium_map_cop

#### 2. Downloading Data

In [ ]:
paths = dag.download(
    cop_products[1],
    output_dir=r"/dss/dsshome1/0C/di54wir/Daten/harvester_eodag",
)

### USGS
#### 1. Searching Data

In [ ]:
bbox = [11.0, 47.9, 11.7, 48.3]

usgs_products = dag.search(
    provider="usgs",
    collection="landsat_etm_c2_l2",
    bbox=bbox,
    #start="2018-06-01",
    #end="2019-06-15",
    limit=10,
    #**{"grid:code": "MGRS-31TFK"}
)

usgs_products

In [ ]:
folium_map_usgs = helper.map_results(True, usgs_products)
folium_map_usgs

#### 2. Download the Data

In [ ]:
paths = dag.download(
    usgs_products[3],
    output_dir=r"/dss/dsshome1/0C/di54wir/Daten/harvester_eodag",
)

### Harvester BB Search vs. CDSE Compatibility

In [ ]:

#url = "https://datahub.creodias.eu/odata/v1"
url = "https://catalogue.dataspace.copernicus.eu/odata/v1"
limit: int = 10

# https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Collection/Name eq 'SENTINEL-2' and contains(Name,'MSIL2A') and ContentDate/Start ge 2018-06-01T00:00:00.000Z and ContentDate/Start lt 2018-06-15T23:59:59.000Z and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'tileId' and att/OData.CSC.StringAttribute/Value eq '32UPU')&$expand=Attributes&$top=10
# https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A') and ContentDate/Start ge 2018-06-01T00:00:00.000Z and ContentDate/Start lt 2018-06-15T23:59:59.000Z and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'tileId' and att/OData.CSC.StringAttribute/Value eq '32UPU')&$expand=Attributes&$top=10

filters: list = [(
#"startswith(Name,'S2') and "
"Collection/Name eq 'SENTINEL-2' and "
#"contains(Name,'MSIL2A') and "
"ContentDate/Start ge 2018-06-01T00:00:00.000Z and "
"ContentDate/Start lt 2018-06-15T23:59:59.000Z and "
"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A') and "
"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'tileId' and att/OData.CSC.StringAttribute/Value eq '32UPU')"
"&$expand=Attributes"
)]

scenes = harvester.search(
    api_url=url,
    max_items=limit,
    filters=filters,
)
scenes[0]

In [ ]:
OData_scenes = {scene["scene_id"].split(".")[0] for scene in scenes}
cop_ids = {prod.properties["id"] for prod in cop_products}

only_in_cop = cop_ids - OData_scenes       
only_in_odata = OData_scenes - cop_ids     
in_both = cop_ids & OData_scenes           

print(f"Both: {len(in_both)}")
print(f"Only Copernicus: {only_in_cop}")
print(f"Only OData: {only_in_odata}")
print("\n"+str(cop_ids))